In [1]:
# !pip install transformers datasets seqeval

## 1. 라이브러리 로드

In [2]:
import pandas as pd
import numpy as np
import urllib.request
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForTokenClassification
from torch.optim import Adam
from seqeval.metrics import f1_score, classification_report
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('사용 디바이스:', device)

사용 디바이스: cuda


## 2. 데이터 로드

In [3]:
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/ukairia777/tensorflow-nlp-tutorial/main/18.%20Fine-tuning%20BERT%20(Cls%2C%20NER%2C%20NLI)/dataset/ner_train_data.csv",
    filename="ner_train_data.csv"
)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/ukairia777/tensorflow-nlp-tutorial/main/18.%20Fine-tuning%20BERT%20(Cls%2C%20NER%2C%20NLI)/dataset/ner_test_data.csv",
    filename="ner_test_data.csv"
)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/ukairia777/tensorflow-nlp-tutorial/main/18.%20Fine-tuning%20BERT%20(Cls%2C%20NER%2C%20NLI)/dataset/ner_label.txt",
    filename="ner_label.txt"
)

train_ner_df = pd.read_csv('ner_train_data.csv')
test_ner_df  = pd.read_csv('ner_test_data.csv')
print('훈련 데이터:', len(train_ner_df), '/ 테스트 데이터:', len(test_ner_df))

훈련 데이터: 81000 / 테스트 데이터: 9000


In [4]:
train_data_sentence = [sent.split() for sent in train_ner_df['Sentence'].values]
test_data_sentence  = [sent.split() for sent in test_ner_df['Sentence'].values]
train_data_label    = [tag.split()  for tag  in train_ner_df['Tag'].values]
test_data_label     = [tag.split()  for tag  in test_ner_df['Tag'].values]

labels = [label.strip() for label in open('ner_label.txt', 'r', encoding='utf-8')]
tag_to_index  = {tag: idx for idx, tag in enumerate(labels)}
index_to_tag  = {idx: tag for idx, tag in enumerate(labels)}
tag_size = len(tag_to_index)
print('개체명 태깅 개수:', tag_size)

개체명 태깅 개수: 29


## 3. 토크나이저 & 전처리

In [5]:
tokenizer = BertTokenizer.from_pretrained('klue/bert-base')

In [6]:
def convert_examples_to_features(examples, labels, max_seq_len, tokenizer):
    input_ids_list, attention_masks_list, token_type_ids_list, labels_list = [], [], [], []

    for example, label in tqdm(zip(examples, labels), total=len(examples)):
        tokens, label_ids = [], []
        for word, label_token in zip(example, label):
            subwords = tokenizer.tokenize(word)
            tokens.extend(subwords)
            label_ids.extend([tag_to_index[label_token]] + [-100] * (len(subwords) - 1))

        # [CLS], [SEP] 고려해서 자르기
        tokens    = tokens[:(max_seq_len - 2)]
        label_ids = label_ids[:(max_seq_len - 2)]

        # [CLS], [SEP] 추가
        tokens    = [tokenizer.cls_token] + tokens + [tokenizer.sep_token]
        label_ids = [-100] + label_ids + [-100]

        input_id      = tokenizer.convert_tokens_to_ids(tokens)
        attention_mask = [1] * len(input_id)
        padding_len   = max_seq_len - len(input_id)

        input_id       = input_id + [tokenizer.pad_token_id] * padding_len
        attention_mask = attention_mask + [0] * padding_len
        token_type_id  = [0] * max_seq_len
        label_ids      = label_ids + [-100] * padding_len

        input_ids_list.append(input_id)
        attention_masks_list.append(attention_mask)
        token_type_ids_list.append(token_type_id)
        labels_list.append(label_ids)

    return (
        torch.tensor(input_ids_list),
        torch.tensor(attention_masks_list),
        torch.tensor(token_type_ids_list),
        torch.tensor(labels_list)
    )

max_seq_len = 128
X_train_ids, X_train_mask, X_train_type, y_train = convert_examples_to_features(
    train_data_sentence, train_data_label, max_seq_len, tokenizer)
X_test_ids,  X_test_mask,  X_test_type,  y_test  = convert_examples_to_features(
    test_data_sentence,  test_data_label,  max_seq_len, tokenizer)

100%|██████████| 9000/9000 [00:02<00:00, 3309.82it/s]


## 4. Dataset & DataLoader

In [7]:
class NERDataset(Dataset):
    def __init__(self, input_ids, attention_mask, token_type_ids, labels):
        self.input_ids      = input_ids
        self.attention_mask = attention_mask
        self.token_type_ids = token_type_ids
        self.labels         = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'token_type_ids': self.token_type_ids[idx],
            'labels':         self.labels[idx]
        }

train_dataset = NERDataset(X_train_ids, X_train_mask, X_train_type, y_train)
test_dataset  = NERDataset(X_test_ids,  X_test_mask,  X_test_type,  y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

## 5. 모델 & 학습

In [8]:
model = BertForTokenClassification.from_pretrained('klue/bert-base', num_labels=tag_size)
model.to(device)
optimizer = Adam(model.parameters(), lr=5e-5)

def evaluate():
    model.eval()
    label_list, pred_list = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            labels         = batch['labels']

            outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                            token_type_ids=token_type_ids)
            preds = torch.argmax(outputs.logits, dim=2).cpu().numpy()
            labels = labels.numpy()

            for label_seq, pred_seq in zip(labels, preds):
                label_tags, pred_tags = [], []
                for l, p in zip(label_seq, pred_seq):
                    if l != -100:
                        label_tags.append(index_to_tag[l])
                        pred_tags.append(index_to_tag[p])
                label_list.append(label_tags)
                pred_list.append(pred_tags)

    score = f1_score(label_list, pred_list, suffix=True)
    print(f'F1: {score*100:.2f}')
    print(classification_report(label_list, pred_list, suffix=True))

epochs = 3
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}'):
        optimizer.zero_grad()
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                        token_type_ids=token_type_ids, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f'Epoch {epoch+1} loss: {total_loss/len(train_loader):.4f}')
    evaluate()

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

c:\AI\envs\pt\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\okss2\.cache\huggingface\hub\models--klue--bert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical

Epoch 1 loss: 0.2721
F1: 85.14
              precision    recall  f1-score   support

         AFW       0.63      0.56      0.59       394
         ANM       0.77      0.70      0.73       701
         CVL       0.83      0.82      0.83      5758
         DAT       0.90      0.92      0.91      2521
         EVT       0.75      0.77      0.76      1094
         FLD       0.77      0.42      0.54       228
         LOC       0.86      0.83      0.85      2126
         MAT       0.25      0.08      0.12        12
         NUM       0.91      0.92      0.91      5590
         ORG       0.86      0.88      0.87      4086
         PER       0.88      0.88      0.88      4426
         PLT       0.50      0.12      0.19        34
         TIM       0.82      0.90      0.86       314
         TRM       0.77      0.71      0.74      1964

   micro avg       0.86      0.85      0.85     29248
   macro avg       0.75      0.68      0.70     29248
weighted avg       0.85      0.85      0.85     2

Epoch 2/3: 100%|██████████| 2532/2532 [17:45<00:00,  2.38it/s]


Epoch 2 loss: 0.1526
F1: 85.92
              precision    recall  f1-score   support

         AFW       0.60      0.59      0.60       394
         ANM       0.73      0.80      0.77       701
         CVL       0.82      0.85      0.83      5758
         DAT       0.92      0.93      0.92      2521
         EVT       0.75      0.78      0.76      1094
         FLD       0.69      0.64      0.67       228
         LOC       0.85      0.86      0.86      2126
         MAT       0.24      0.42      0.30        12
         NUM       0.91      0.93      0.92      5590
         ORG       0.87      0.87      0.87      4086
         PER       0.89      0.90      0.89      4426
         PLT       0.57      0.24      0.33        34
         TIM       0.83      0.94      0.88       314
         TRM       0.74      0.76      0.75      1964

   micro avg       0.85      0.87      0.86     29248
   macro avg       0.74      0.75      0.74     29248
weighted avg       0.85      0.87      0.86     2

Epoch 3/3: 100%|██████████| 2532/2532 [17:42<00:00,  2.38it/s]


Epoch 3 loss: 0.1018
F1: 86.57
              precision    recall  f1-score   support

         AFW       0.68      0.63      0.65       394
         ANM       0.76      0.79      0.78       701
         CVL       0.84      0.85      0.84      5758
         DAT       0.93      0.92      0.93      2521
         EVT       0.76      0.76      0.76      1094
         FLD       0.69      0.62      0.65       228
         LOC       0.87      0.85      0.86      2126
         MAT       0.25      0.33      0.29        12
         NUM       0.91      0.93      0.92      5590
         ORG       0.89      0.87      0.88      4086
         PER       0.89      0.90      0.90      4426
         PLT       0.44      0.21      0.28        34
         TIM       0.87      0.91      0.89       314
         TRM       0.78      0.74      0.76      1964

   micro avg       0.87      0.87      0.87     29248
   macro avg       0.75      0.74      0.74     29248
weighted avg       0.87      0.87      0.87     2

## 6. 예측

In [9]:
def ner_prediction(sentence):
    model.eval()
    words = sentence.split()
    tokens, label_mask = [], []
    for word in words:
        subwords = tokenizer.tokenize(word)
        tokens.extend(subwords)
        label_mask.extend([0] + [-100] * (len(subwords) - 1))

    tokens    = [tokenizer.cls_token] + tokens[:(max_seq_len-2)] + [tokenizer.sep_token]
    label_mask = [-100] + label_mask[:(max_seq_len-2)] + [-100]
    input_id  = tokenizer.convert_tokens_to_ids(tokens)

    input_ids      = torch.tensor([input_id]).to(device)
    attention_mask = torch.tensor([[1]*len(input_id)]).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    preds = torch.argmax(outputs.logits, dim=2)[0].cpu().numpy()

    result = []
    word_idx = 0
    for mask, pred in zip(label_mask, preds):
        if mask != -100:
            result.append((words[word_idx], index_to_tag[pred]))
            word_idx += 1
    return result

sent1 = '오리온스는 리그 최정상급 포인트가드 김동훈을 앞세우는 빠른 공수전환이 돋보이는 팀이다'
sent2 = '하이신사에 속한 섬들도 위로 솟아 있는데 타인은 살고 있어요'
print(ner_prediction(sent1))
print(ner_prediction(sent2))

[('오리온스는', 'ORG-B'), ('리그', 'O'), ('최정상급', 'O'), ('포인트가드', 'CVL-B'), ('김동훈을', 'PER-B'), ('앞세우는', 'O'), ('빠른', 'O'), ('공수전환이', 'O'), ('돋보이는', 'O'), ('팀이다', 'O')]
[('하이신사에', 'LOC-B'), ('속한', 'O'), ('섬들도', 'O'), ('위로', 'O'), ('솟아', 'O'), ('있는데', 'O'), ('타인은', 'O'), ('살고', 'O'), ('있어요', 'O')]
